In [1]:
#!pip install --upgrade yfinance

In [2]:
import requests, os
from datetime import date
import pandas as pd

### Import via Companies MarketCap

In [3]:
# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


Arquivo salvo em: ../data/raw/top_companies_revenue_20261102.csv
Tamanho: 777,772 bytes


### Padronização e Ajuste de Nomenclatura do Dataset

In [4]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "ticker_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)


from bs4 import BeautifulSoup
def clean_html(text):
    if pd.isna(text):
        return text
    return BeautifulSoup(text, "html.parser").get_text()

df["company_name"] = df["company_name"].apply(clean_html)

#check for non unique tickers and print the ones that are not unique 
non_unique_tickers = df[df.duplicated(subset=["ticker_symbol"], keep=False)]
if not non_unique_tickers.empty:
    print("Non-unique tickers found:")
    print(non_unique_tickers[["ticker_symbol", "company_name"]])


print("Número de empresas únicas:", df["ticker_symbol"].nunique())
print("Total de receita (TTM) em USD:", f"${df['revenue_ttm'].sum() / 1e12:.2f} tri")

display(df.head(1))

Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20261102.csv
Número de empresas únicas: 11395
Total de receita (TTM) em USD: $67.03 tri


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country
0,1,Amazon,AMZN,742776000000,240.35,United States


### Limpar Dataset seguindo escopo estabelecido

In [5]:

# retirar empresas com receita negativa ou nula
print("Retirando empresas com receita negativa ou nula...")
df = df[df["revenue_ttm"] > 0]

# listar linhas que estao duplicadas no subset de colunas "company_name" e "revenue_ttm"
duplicated_rows = df[df.duplicated(subset=["company_name", "revenue_ttm"], keep=False)]
print("Linhas duplicadas:")
print(duplicated_rows)

# drop_duplicates remove linhas repetidas; subset define o que conta como "igual"
# Usar nome + receita evita apagar homônimos diferentes (Castellum, First Bancorp)
df = df.drop_duplicates(subset=["company_name", "revenue_ttm"])
print("Empresas duplicadas removidas")

print()

print("Número de empresas únicas após remoção de duplicatas e receita negativa ou nula:", df["ticker_symbol"].nunique())
print("Total de receita (TTM) em USD:", f"${df['revenue_ttm'].sum() / 1e12:.2f} tri")

Retirando empresas com receita negativa ou nula...
Linhas duplicadas:
     rank   company_name ticker_symbol  revenue_ttm  price_usd company_country
189   190  SK Innovation     096775.KS  62996237095    49.1254     South Korea
190   191  SK Innovation     096770.KS  62996237095    99.1563     South Korea
Empresas duplicadas removidas

Número de empresas únicas após remoção de duplicatas e receita negativa ou nula: 10666
Total de receita (TTM) em USD: $66.98 tri


### Informações Preliminares do Dataset

In [6]:
import pandas as pd

PIB = 118_350_166_039_396  # Banco Mundial, 2025, USD correntes

# numero de empresas com receita positiva
num_companies = len(df)
print(f"Número de empresas com receita positiva: {num_companies}")

# Receita total após a limpeza
total = df["revenue_ttm"].sum()

print(f"Total: {total/1e12:.2f} tri")
print(f"% do PIB: {total/PIB*100:.1f}%")

# 1) Soma do top 500 por receita: deve ficar perto dos ~41,7 tri da Fortune 500
#    nlargest(500, coluna) devolve as 500 linhas com maior valor na coluna
top500 = df.nlargest(500, "revenue_ttm")["revenue_ttm"].sum()
print(f"Top 500: {top500/1e12:.1f} tri")

# 2) Os 20 maiores: confira à mão com a Fortune (Walmart, Amazon, State Grid, etc.)
print(df.nlargest(20, "revenue_ttm")[["company_name", "revenue_ttm"]])

# 3) Duplicidades: nomes repetidos podem indicar ADR/dupla listagem
#    duplicated(keep=False) marca TODAS as linhas repetidas, não só a segunda
dups = df[df.duplicated(subset="company_name", keep=False)].sort_values("company_name")
print(len(dups))

# 4) Peso dos maiores: se poucas linhas explicam muito do total, olhe-as primeiro
print(df["revenue_ttm"].describe())

Número de empresas com receita positiva: 10667
Total: 66.98 tri
% do PIB: 56.6%
Top 500: 39.3 tri
                            company_name   revenue_ttm
0                                 Amazon  742776000000
1                                Walmart  735840000000
2                           Saudi Aramco  487870749993
3                                  Apple  466823000000
4                           UnitedHealth  450129000000
5                      Alphabet (Google)  445867000000
6                                Sinopec  444749587530
7                     Berkshire Hathaway  420276000000
8                             PetroChina  419928669654
9                             CVS Health  415086000000
10                              McKesson  410983000000
11                            Volkswagen  374700624687
12                           Exxon Mobil  361060000000
13                               Cencora  332771323000
14                                Toyota  332273722835
15                    

### Enriquecimento com Yahoo Finance

In [7]:
import os, time
import pandas as pd
from functools import partial
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from IPython.display import display, clear_output

import importlib
import src.yf_pipeline as yfp   # "src" é subpasta da pasta atual do notebook

importlib.reload(yfp)                    # recarrega o .py editado sem reiniciar o kernel

# ---------------- CONFIG ----------------
OUTPUT_PATH = "../data/interim/01_data_after_yfinance.csv"   # checkpoint
CACHE_PATH  = "../data/interim/yf_cache.csv"
CHUNK_SIZE, WAIT_TIME, MAX_WORKERS = 100, 5, 10
PAUSA_RATE_LIMIT, JITTER_MAX = 60, 3.0

# ---------------- 0. TRATAR ERROS CONHECIDOS ----------------
# Dicionário de correção de tickers que estão errados para os certos

tickers_corrigidos = {
    "TATAMOTORS.NS": "TMPV.NS",   # renomeado em 24/10/2025 após o demerger (a divisão de veículos comerciais virou TMCV.NS, empresa nova)
    "PHNX.L":        "SDLF.L",    #  ticker antigo PHNX.L (anteriormente Phoenix Group Holdings plc) foi renomeado e agora opera sob o nome Standard Life plc, com o novo ticker principal na London Stock Exchange (LSE) sendo SDLF.L
    "ML.PA":         "ML.PA",     # sem alteração
    "FISV":          "FISV",      # sem alteração (ver nota abaixo)
    "0489.HK":       "7489.HK",   # Dongfeng saiu da bolsa; a Voyah (ex-subsidiária) listou em 19/03/2026
    "XEKT.MC":       "ELEKTRA.MX",# Latibex excluiu a ação; a BMV suspendeu a negociação em 30/09/2025
    "7205.T":        "543A.T",    # Hino deslistada em mar/abr 2026; virou subsidiária da ARCHION
    #"QVCGA":         None,        # Chapter 11 em abr/2026; Nasdaq suspendeu em 24/04/2026
    "IHC.AE":        "IHC.AE",    # sem alteração (não confirmei o formato direto no Yahoo)
    "SGH.AX":        "SGH.AX",    # sem alteração (o problema era o \n no nome)
    "SEM":           "SEM",       # sem alteração
    #"WG.L":          None,        # adquirida pela Sidara, deslistada da LSE em 11/03/2026
    "MQG.AX":        "MQG.AX",    # sem alteração
    #"YELLQ":         None,        # falência (2023), sem cotação ativa
    #"7105.T":        None,        # tender offer + deslistagem em 27/04/2026
    "LTIM.NS":       "LTM.NS",    # símbolo mudou em 27/02/2026; razão social virou LTM Limited
    #"2692.T":        None,        # cash-out pela Itochu, deslistada em 19/05/2026
    "ALBH.BH":       "ALBH.L",    # Alba é dual-listed; a cotação na LSE é via GDR (formato Yahoo não confirmado)
    "FDP":           "FDP",       # sem alteração
    "ASGN":          "ASGN",      # sem alteração
}

#update ticker_symbol column with corrected tickers
df["ticker_symbol"] = df["ticker_symbol"].replace(tickers_corrigidos)

# ---------------- 1. PREPARAR ----------------
yfp.garantir_colunas(df)                     # cria colunas yf_* que faltarem (in-place)
if os.path.exists(OUTPUT_PATH):              # retoma do checkpoint
    yfp.aplicar_resultados(df, pd.read_csv(OUTPUT_PATH))
cache = yfp.carregar_cache(CACHE_PATH)       # dict em memória
print(f"Cache: {len(cache)} tickers")

# ---------------- 2. PENDENTES ----------------
pendentes = yfp.listar_pendentes(df, reprocessar_not_found=True, ordem_reversa=True)
print(f"Total: {len(df)} | Pendentes: {len(pendentes)}")

# ---------------- 3. LOOP (visível aqui) ----------------
# partial "fixa" cache e parâmetros: executor.map só aceita funções de 1 argumento
resolver = partial(yfp.resolver_ticker, cache=cache,
                   pausa_segundos=PAUSA_RATE_LIMIT, jitter_max=JITTER_MAX)

for i in tqdm(range(0, len(pendentes), CHUNK_SIZE), desc="Lotes"):
    clear_output()
    chunk = pendentes[i:i + CHUNK_SIZE]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        resultados = list(ex.map(resolver, chunk))      # 1 chamada por ticker

    yfp.aplicar_resultados(df, resultados)              # grava no df, por ticker
    yfp.atualizar_cache(cache, resultados)              # sucessos novos entram no dict
    yfp.salvar_cache(cache, CACHE_PATH)                 # dict -> CSV
    yfp.salvar_csv(df, OUTPUT_PATH)                     # checkpoint por lote

    # só espera se algum ticker do lote realmente foi ao Yahoo
    if any(r["origem"] == "yahoo" for r in resultados):
        time.sleep(WAIT_TIME)

# ---------------- 4. AJUSTES E VISUALIZAÇÃO ----------------

# fazer limpeza pós resolver: se o yf_status é "success" mas o website está ausente, então o ticker não foi resolvido de fato
df = pd.read_csv(OUTPUT_PATH)

# Máscara 1: linhas onde o yf_status não foi sucesso
sucesso = df["yf_status"] == "success"
# Máscara 2: website ausente (NaN real) ou string vazia
website_vazio = df["website"].isna() | (df["website"] == "")

# Combina as duas condições com & (E lógico bit a bit — para Series,
# use sempre & / | / ~ em vez de and/or/not, que não funcionam elemento a elemento)
mask = sucesso & website_vazio
df.loc[mask, "yf_status"] = "error: ticker not resolved"
# opcional: conferir quantas linhas foram afetadas
print(mask.sum())
df.to_csv(OUTPUT_PATH, index=False)


clear_output()
print(f"Total de tickers: {len(df)} | Resolvidos: {len(df[df['yf_status'] == 'success'])} | Não Resolvidos: {len(df[df['yf_status'] == 'error: ticker not resolved'])} | Cache: {len(cache)}")

display(df.head(10))

Total de tickers: 10667 | Resolvidos: 10324 | Não Resolvidos: 342 | Cache: 10881


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
0,1,Amazon,AMZN,742776000000,240.350000,United States,https://www.amazon.com,"Amazon.com, Inc. engages in the retail sale of...",Consumer Cyclical,Internet Retail,United States,Seattle,success
1,2,Walmart,WMT,735840000000,123.510000,United States,https://www.stock.walmart.com,Walmart Inc. engages in the operation of retai...,Consumer Defensive,Discount Stores,United States,Bentonville,success
2,3,Saudi Aramco,2222.SR,487870749993,6.958330,Saudi Arabia,https://www.aramco.com,Saudi Arabian Oil Company operates as an integ...,Energy,Oil & Gas Integrated,Saudi Arabia,Dhahran,success
3,4,Apple,AAPL,466823000000,344.630000,United States,https://www.apple.com,"Apple Inc. designs, manufactures, and markets ...",Technology,Consumer Electronics,United States,Cupertino,success
4,5,UnitedHealth,UNH,450129000000,406.970000,United States,https://www.unitedhealthgroup.com,UnitedHealth Group Incorporated operates as a ...,Healthcare,Healthcare Plans,United States,Eden Prairie,success
5,6,Alphabet (Google),GOOG,445867000000,278.220000,United States,https://abc.xyz,Alphabet Inc. offers various products and plat...,Communication Services,Internet Content & Information,United States,Mountain View,success
6,7,Sinopec,600028.SS,444749587530,0.883922,China,https://www.sinopec.com/listco,"China Petroleum & Chemical Corporation, an ene...",Energy,Oil & Gas Integrated,China,Beijing,success
7,8,Berkshire Hathaway,BRK-B,420276000000,542.490000,United States,https://www.berkshirehathaway.com,"Berkshire Hathaway Inc., together with its sub...",Financial Services,Insurance - Diversified,United States,Omaha,success
8,9,PetroChina,0857.HK,419928669654,1.377810,China,https://www.petrochina.com.cn,"PetroChina Company Limited, together with its ...",Energy,Oil & Gas Integrated,China,Beijing,success
9,10,CVS Health,CVS,415086000000,90.250000,United States,https://www.cvshealth.com,CVS Health Corporation provides health solutio...,Healthcare,Healthcare Plans,United States,Woonsocket,success


### Análise sobre lacunas de informação

In [8]:
import pandas as pd

df = pd.read_csv(OUTPUT_PATH)

# Status que você quer medir (ajuste se o texto for diferente)
STATUS_ERRO = "error: ticker not resolved"

total_geral = df["revenue_ttm"].sum()

for n in (500, 1000, 2000, 5000, df.shape[0]):
      topn = df.nlargest(n, "revenue_ttm")

      mask_nao_resolvidas = (
            (topn["yf_status"] == STATUS_ERRO)
            | topn["sector"].isna()
            | topn["industry"].isna()
      )
      nao_resolvidas = topn[mask_nao_resolvidas]

      receita_topn = topn["revenue_ttm"].sum()
      receita_nr = nao_resolvidas["revenue_ttm"].sum()

      print(f"Top {n}:")
      print(f"  Não resolvidas: {len(nao_resolvidas)} de {n} ({len(nao_resolvidas) / n * 100:.1f}%)")
      print(f"  Receita do top {n}: {receita_topn / 1e12:.2f} tri "
              f"({receita_topn / total_geral * 100:.1f}% do total)")
      print(f"  Receita não resolvida: {receita_nr / 1e12:.3f} tri "
              f"({receita_nr / receita_topn * 100:.2f}% do top {n})")

      if not nao_resolvidas.empty:
            display(nao_resolvidas)
    

Top 500:
  Não resolvidas: 1 de 500 (0.2%)
  Receita do top 500: 39.28 tri (58.6% do total)
  Receita não resolvida: 0.031 tri (0.08% do top 500)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
421,423,Michelin,ML.PA,30533561783,36.9476,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 1000:
  Não resolvidas: 2 de 1000 (0.2%)
  Receita do top 1000: 48.67 tri (72.7% do total)
  Receita não resolvida: 0.051 tri (0.11% do top 1000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
421,423,Michelin,ML.PA,30533561783,36.9476,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
645,647,Fiserv,FISV,20866000000,50.0200,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 2000:
  Não resolvidas: 7 de 2000 (0.4%)
  Receita do top 2000: 57.37 tri (85.7% do total)
  Receita não resolvida: 0.096 tri (0.17% do top 2000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
421,423,Michelin,ML.PA,30533561783,36.9476,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
645,647,Fiserv,FISV,20866000000,50.0200,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1163,1165,Grupo Elektra,ELEKTRA.MX,11317736000,20.2458,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1369,1371,QVC Group,QVCGA,9231000000,0.3400,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1410,1412,Kaspi.kz Joint Stock Company,KKS.F,8910057578,91.2155,Kazakhstan,https://ir.kaspi.kz,"Joint Stock Company Kaspi.kz, together with it...",NaN,NaN,Kazakhstan,Almaty,success
1571,1573,International Holding Company,IHC.AE,7747610218,92.5800,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1651,1653,Seven Group Holdings\r\n (SGH),SGH.AX,7320175531,29.4687,Australia,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 5000:
  Não resolvidas: 63 de 5000 (1.3%)
  Receita do top 5000: 65.20 tri (97.3% do total)
  Receita não resolvida: 0.232 tri (0.36% do top 5000)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
421,423,Michelin,ML.PA,30533561783,36.9476,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
645,647,Fiserv,FISV,20866000000,50.0200,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1163,1165,Grupo Elektra,ELEKTRA.MX,11317736000,20.2458,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1369,1371,QVC Group,QVCGA,9231000000,0.3400,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1410,1412,Kaspi.kz Joint Stock Company,KKS.F,8910057578,91.2155,Kazakhstan,https://ir.kaspi.kz,"Joint Stock Company Kaspi.kz, together with it...",NaN,NaN,Kazakhstan,Almaty,success
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4801,4803,Nilfisk Holding,NLFSK.CO,1147368156,21.9239,Denmark,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4819,4821,Bed Bath & Beyond,BBBY,1139531000,4.3500,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4838,4840,Shoe Carnival,SCVL,1128339000,17.4300,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
4902,4904,Itaú CorpBanca,ITCLY,1087155900,3.1000,Chile,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


Top 10667:
  Não resolvidas: 353 de 10667 (3.3%)
  Receita do top 10667: 66.98 tri (100.0% do total)
  Receita não resolvida: 0.291 tri (0.43% do top 10667)


,rank,company_name,ticker_symbol,revenue_ttm,price_usd,company_country,website,longBusinessSummary,sector,industry,country,city,yf_status
421,423,Michelin,ML.PA,30533561783,36.9476,France,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
645,647,Fiserv,FISV,20866000000,50.0200,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1163,1165,Grupo Elektra,ELEKTRA.MX,11317736000,20.2458,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1369,1371,QVC Group,QVCGA,9231000000,0.3400,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
1410,1412,Kaspi.kz Joint Stock Company,KKS.F,8910057578,91.2155,Kazakhstan,https://ir.kaspi.kz,"Joint Stock Company Kaspi.kz, together with it...",NaN,NaN,Kazakhstan,Almaty,success
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10627,10629,Advent Technologies,ADN,49000,0.9185,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10628,10630,Zoom Telephonics,MINM,45118,3.6100,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10636,10638,Forza X1,FRZA,18559,0.2000,United States,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved
10638,10640,Lucy Scientific Discovery,LSDI,16732,0.5139,Canada,NaN,NaN,NaN,NaN,NaN,NaN,error: ticker not resolved


In [9]:
# print list of company_name, ticker_symbol, for companies that were not resolved
not_resolved = df[df['yf_status'] == 'error: ticker not resolved']
for _, row in not_resolved.iterrows():
    print(f"Company: {row['company_name']}, Ticker: {row['ticker_symbol']}")


Company: Michelin, Ticker: ML.PA
Company: Fiserv, Ticker: FISV
Company: Grupo Elektra, Ticker: ELEKTRA.MX
Company: QVC Group, Ticker: QVCGA
Company: International Holding Company, Ticker: IHC.AE
Company: Seven Group Holdings
 (SGH), Ticker: SGH.AX
Company: Select Medical Holdings, Ticker: SEM
Company: John Wood Group, Ticker: WG.L
Company: Macquarie Group Limited, Ticker: MQG.AX
Company: Yellow Corporation, Ticker: YELLQ
Company: Mitsubishi Logisnext, Ticker: 7105.T
Company: ITOCHU-SHOKUHIN, Ticker: 2692.T
Company: Aluminium Bahrain (Alba), Ticker: ALBH.L
Company: Fresh Del Monte Produce, Ticker: FDP
Company: ASGN, Ticker: ASGN
Company: Unibel S.A., Ticker: UNBL.PA
Company: GOL Airlines, Ticker: GOLL54.SA
Company: Absa Bank, Ticker: ABSP.JO
Company: SCSK Corporation, Ticker: 9719.T
Company: Light & Wonder, Ticker: LNW
Company: Swiss Steel, Ticker: STLN.SW
Company: Vertex Energy, Ticker: VTNR
Company: Trinseo, Ticker: TSE
Company: Lionsgate Studios, Ticker: LION
Company: CCC S.A., Ticke